# Steps for Colab


1.   Add dataset.jsonl in /content/
2.   Run cells until huggingface login() prompt, and provide your authentication token (recommended, but can be skipped)
3.   Run remaining cells. Last cell will prompt the browser to download the .zip file for the LoRA weights


In [ ]:
!pip install --upgrade --no-cache-dir transformers
!pip install -q accelerate peft bitsandbytes datasets
!pip install -U bitsandbytes
!pip install -U transformers
!accelerate config default

In [ ]:
# Use to check if the dataset looks as expected. For example:
# {"prompt": "Translate the following to American Sign Language (ASL) structure:\n I’ve never heard of that!", "completion": "NEVER EAR THAT!"}

with open("asl_dataset.jsonl", "r", encoding="utf-8") as f:
    for _ in range(5):
        print(f.readline())

#Login with Hugging Face API for authenticated access to the base model. Not necessary, but recommended for more bandwidth

In [ ]:
# This cell is optional, but recommended to speed up base-model download speed
!pip install huggingface_hub
!accelerate config default
import os
from huggingface_hub import login

login()


#Setup config

In [ ]:
# import the base-model and the tokenizer
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

model_name = "mistralai/Mistral-7B-Instruct-v0.2"

# Config for 4-bit Quantized LoRA (QLoRA)
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

# Loads tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Loads model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=quant_config
)


#Setup LoRA config and load the base-model from huggingface

In [ ]:
# imports LoRA and configures it correctly for our base-model

from peft import LoraConfig, get_peft_model

# r (rank) - increases training capacity and VRAM usage
# alpha - scales the strength of the fine-tuned adjustments. Common heuristic is r*2
# target_modules - specify which parts of the model you want to apply LoRA adapters to
# dropout - randomly parts of LoRA activations to 0 to prevent overfitting. Recommended 0 - 0.1
# bias - bias type for LoRA. "none" is recommended, but can be "all" and "lora_only" too
# task_type - attribute dependency for get_peft_model(). Recommended "CASUAL_LM"

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # specific LoRA params for Mistral
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

print("LoRA-config added")

#Load dataset and apply train-test splitting

In [ ]:
import pandas as pd
from datasets import Dataset

# load dataset and perform train-test splitting to avoid model overfitting
# read more: https://www.kaggle.com/code/mgalarny/train-test-split-tutorial-in-too-much-detail

df = pd.read_json("/content/asl_dataset.jsonl", lines=True)
dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.1)

print("Example:", dataset["train"][0])


#Format and tokenize dataset

In [ ]:
def format_and_tokenize(example):
    messages = [
        {"role": "user", "content": example["prompt"]},
        {"role": "assistant", "content": example["completion"]}
    ]
    full_text = tokenizer.apply_chat_template(messages, tokenize=False)

    tokenized = tokenizer(
        full_text,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

    return {
        "input_ids": tokenized["input_ids"][0],
        "labels": tokenized["input_ids"][0]
    }

tokenized_dataset = dataset.map(format_and_tokenize)

print("Tokenization with full chat-template done. Example:")
print(tokenized_dataset["train"][0])



#Create LoRA weights and download results from browser

In [ ]:
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

training_args = TrainingArguments(
    output_dir="./mistral-asl-instruct",
    per_device_train_batch_size=8,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_steps=50,
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

# Save LoRA-weights after training (This is crucial to include in the same as trainer.train() (after trainer.train as we have done here) as Colab runtime is extremely unpredictable and will randomly disconnect you)
trainer.model.save_pretrained("./mistral-asl-instruct")
tokenizer.save_pretrained("./mistral-asl-instruct")

print("The model and tokenizer is saved")

In [ ]:
import shutil
from google.colab import files

# Create the linked path folder into a zip
shutil.make_archive("mistral-asl-instruct", "zip", "./mistral-asl-instruct")

# Downloads zip file from browser
files.download("mistral-asl-instruct.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>